## 1. Setup and Imports

# How Does Neural Network Depth Affect Learning?
## A Tutorial on MLPs and the Power of Depth

In this notebook, we explore how the depth of a Multi-Layer Perceptron (MLP) affects its ability to learn and generalize. We'll train networks of varying depths on the MNIST dataset and visualize how performance changes.

### Key Question
Does a deeper network always learn better? Let's find out!

### References
- LeCun et al. (1998) - Gradient-based learning applied to document recognition
- Bengio et al. (2013) - On the difficulty of training deep feedforward neural networks
- He et al. (2016) - Deep Residual Learning for Image Recognition (ResNet)

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 2. Load and Prepare Data

We use the digits dataset (8x8 images of handwritten digits 0-9). This is smaller than MNIST but sufficient for our purposes.

In [12]:
# Load digits dataset
digits = load_digits()
X, y = digits.data, digits.target

print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Feature dimension: {X.shape[1]}")

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Dataset shape: (1797, 64)
Number of classes: 10
Feature dimension: 64

Training set size: 1437
Test set size: 360


## 3. Visualize Sample Data

Let's look at what we're working with - handwritten digits.

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Label: {y[i]}')
    ax.axis('off')
plt.tight_layout()
plt.savefig('figures/sample_digits.png', dpi=150, bbox_inches='tight')
plt.show()

print("Sample digits visualization saved.")

## 4. Define Network Architectures of Varying Depth

We'll create MLPs with different depths:
- **Shallow (1-2 hidden layers)**: Quick to train, limited capacity
- **Medium (3-4 hidden layers)**: Balanced approach
- **Deep (5-10+ hidden layers)**: More complex, harder to train

All networks have the same hidden layer width (128 neurons) to isolate the effect of depth.

In [14]:
# Define network architectures
# Format: (depth, architecture)
architectures = {
    'Depth 1': (128,),                                    # 1 hidden layer
    'Depth 2': (128, 128),                                # 2 hidden layers
    'Depth 3': (128, 128, 128),                           # 3 hidden layers
    'Depth 4': (128, 128, 128, 128),                      # 4 hidden layers
    'Depth 5': (128, 128, 128, 128, 128),                 # 5 hidden layers
    'Depth 7': (128, 128, 128, 128, 128, 128, 128),       # 7 hidden layers
    'Depth 10': (128,) * 10,                              # 10 hidden layers
}

print("Network architectures defined:")
for name, arch in architectures.items():
    print(f"{name}: {len(arch)} hidden layers with {arch[0]} neurons each")

Network architectures defined:
Depth 1: 1 hidden layers with 128 neurons each
Depth 2: 2 hidden layers with 128 neurons each
Depth 3: 3 hidden layers with 128 neurons each
Depth 4: 4 hidden layers with 128 neurons each
Depth 5: 5 hidden layers with 128 neurons each
Depth 7: 7 hidden layers with 128 neurons each
Depth 10: 10 hidden layers with 128 neurons each


## 5. Train Networks and Record Performance

This may take a few minutes. We're training each network and tracking:
- Training accuracy
- Test accuracy
- Training loss over epochs

In [15]:
# Dictionary to store results
results = {}
training_losses = {}

print("Training networks of varying depths...\n")

for name, hidden_layers in architectures.items():
    print(f"Training {name}...", end=" ")
    
    # Create and train the model
    mlp = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation='relu',
        solver='adam',
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=50,
        verbose=0
    )
    
    # Train the model
    mlp.fit(X_train, y_train)
    
    # Get predictions
    y_train_pred = mlp.predict(X_train)
    y_test_pred = mlp.predict(X_test)
    
    # Calculate accuracies
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    
    # Store results
    results[name] = {
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'n_layers': len(hidden_layers),
        'n_iters': mlp.n_iter_
    }
    training_losses[name] = mlp.loss_curve_
    
    print(f"Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

print("\nTraining complete!")

Training networks of varying depths...

Training Depth 1... Train Acc: 0.9965, Test Acc: 0.9694
Training Depth 2... Train Acc: 0.9986, Test Acc: 0.9694
Training Depth 3... Train Acc: 0.9972, Test Acc: 0.9694
Training Depth 4... Train Acc: 0.9979, Test Acc: 0.9694
Training Depth 5... Train Acc: 0.9930, Test Acc: 0.9667
Training Depth 7... Train Acc: 0.9965, Test Acc: 0.9639
Training Depth 10... Train Acc: 0.9889, Test Acc: 0.9528

Training complete!


## 6. Display Results as Table

In [16]:
# Create results dataframe for easy viewing
import pandas as pd

results_df = pd.DataFrame(results).T
results_df = results_df.round(4)

print("\n=== PERFORMANCE SUMMARY ===")
print(results_df)
print(f"\nBest test accuracy: {results_df['test_accuracy'].max():.4f}")
print(f"Best performing depth: {results_df.loc[results_df['test_accuracy'].idxmax(), 'n_layers']:.0f} layers")


=== PERFORMANCE SUMMARY ===
          train_accuracy  test_accuracy  n_layers  n_iters
Depth 1           0.9965         0.9694       1.0    121.0
Depth 2           0.9986         0.9694       2.0    101.0
Depth 3           0.9972         0.9694       3.0     66.0
Depth 4           0.9979         0.9694       4.0    107.0
Depth 5           0.9930         0.9667       5.0     61.0
Depth 7           0.9965         0.9639       7.0     61.0
Depth 10          0.9889         0.9528      10.0     61.0

Best test accuracy: 0.9694
Best performing depth: 1 layers


## 7. Visualize: Accuracy vs Network Depth

The main finding: how does accuracy change as we go deeper?

In [ ]:
# Extract data for plotting
depths = [results[name]['n_layers'] for name in results.keys()]
train_accs = [results[name]['train_accuracy'] for name in results.keys()]
test_accs = [results[name]['test_accuracy'] for name in results.keys()]

# Create the plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(depths, train_accs, marker='o', linewidth=2.5, markersize=8, label='Training Accuracy', color='#2ecc71')
ax.plot(depths, test_accs, marker='s', linewidth=2.5, markersize=8, label='Test Accuracy', color='#e74c3c')

ax.set_xlabel('Network Depth (Number of Hidden Layers)', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('How Network Depth Affects Learning Performance', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='lower right')
ax.set_xticks(depths)
ax.set_ylim([0.7, 1.02])

# Add value labels on points
for i, (d, ta, tea) in enumerate(zip(depths, train_accs, test_accs)):
    ax.text(d, ta + 0.01, f'{ta:.3f}', ha='center', fontsize=9)
    ax.text(d, tea - 0.03, f'{tea:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('figures/accuracy_vs_depth.png', dpi=150, bbox_inches='tight')
plt.show()

print("Accuracy plot saved.")

## 8. Visualize: Training Curves

How fast do networks of different depths converge?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.viridis(np.linspace(0, 1, len(training_losses)))

for idx, (name, losses) in enumerate(training_losses.items()):
    ax.plot(losses, label=name, linewidth=2, color=colors[idx], alpha=0.8)

ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax.set_ylabel('Loss', fontsize=12, fontweight='bold')
ax.set_title('Training Loss Over Epochs for Different Network Depths', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("Training curves plot saved.")

## 9. Analyze the Gap Between Training and Test Accuracy

Overfitting is a concern with deeper networks. Let's visualize the generalization gap.

In [ ]:
# Calculate generalization gap
gaps = [results[name]['train_accuracy'] - results[name]['test_accuracy'] for name in results.keys()]

fig, ax = plt.subplots(figsize=(10, 6))

colors_gap = ['#27ae60' if g < 0.05 else '#f39c12' if g < 0.1 else '#e74c3c' for g in gaps]

bars = ax.bar(range(len(results)), gaps, color=colors_gap, alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_xlabel('Network Depth', fontsize=12, fontweight='bold')
ax.set_ylabel('Overfitting Gap (Train Acc - Test Acc)', fontsize=12, fontweight='bold')
ax.set_title('Overfitting Gap vs Network Depth', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(results)))
ax.set_xticklabels(results.keys(), rotation=45, ha='right')
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, (bar, gap) in enumerate(zip(bars, gaps)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{gap:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('figures/overfitting_gap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Overfitting gap plot saved.")

## 10. Key Insights and Conclusions

Let's summarize what we learned:

In [20]:
print("="*60)
print("KEY INSIGHTS: Network Depth and Learning")
print("="*60)

best_idx = np.argmax(test_accs)
best_depth = depths[best_idx]
best_acc = test_accs[best_idx]

print(f"\n1. OPTIMAL DEPTH:")
print(f"   Best test accuracy: {best_acc:.4f} at depth {best_depth}")

print(f"\n2. SHALLOW VS DEEP:")
shallow_acc = test_accs[0]
improvement = (best_acc - shallow_acc) / shallow_acc * 100
print(f"   Depth 1 accuracy: {shallow_acc:.4f}")
print(f"   Best accuracy:    {best_acc:.4f}")
print(f"   Improvement:      {improvement:.2f}%")

print(f"\n3. OVERFITTING TREND:")
avg_gap_shallow = np.mean(gaps[:2])
avg_gap_deep = np.mean(gaps[-2:])
print(f"   Avg gap (depths 1-2):  {avg_gap_shallow:.4f}")
print(f"   Avg gap (depths 7-10): {avg_gap_deep:.4f}")
if avg_gap_deep > avg_gap_shallow:
    print(f"   → Deeper networks show MORE overfitting")
else:
    print(f"   → Deeper networks show LESS overfitting")

print(f"\n4. CONVERGENCE:")
fastest = min(results.items(), key=lambda x: x[1]['n_iters'])
slowest = max(results.items(), key=lambda x: x[1]['n_iters'])
print(f"   Fastest converging: {fastest[0]} ({fastest[1]['n_iters']} epochs)")
print(f"   Slowest converging: {slowest[0]} ({slowest[1]['n_iters']} epochs)")

print(f"\n" + "="*60)
print("CONCLUSION:")
print("="*60)
print(f"\nDepth DOES improve performance on this task, but there are")
print(f"diminishing returns. A depth of {best_depth} layers is optimal,")
print(f"suggesting that for this relatively simple task, very deep")
print(f"networks are not necessary. This is a common pattern in ML!")
print()

KEY INSIGHTS: Network Depth and Learning

1. OPTIMAL DEPTH:
   Best test accuracy: 0.9694 at depth 1

2. SHALLOW VS DEEP:
   Depth 1 accuracy: 0.9694
   Best accuracy:    0.9694
   Improvement:      0.00%

3. OVERFITTING TREND:
   Avg gap (depths 1-2):  0.0281
   Avg gap (depths 7-10): 0.0344
   → Deeper networks show MORE overfitting

4. CONVERGENCE:
   Fastest converging: Depth 5 (61 epochs)
   Slowest converging: Depth 1 (121 epochs)

CONCLUSION:

Depth DOES improve performance on this task, but there are
diminishing returns. A depth of 1 layers is optimal,
suggesting that for this relatively simple task, very deep
networks are not necessary. This is a common pattern in ML!



## 11. Challenge for Learners

Try these experiments yourself:

1. **Vary width instead of depth**: Keep depth at 3 but try hidden layer sizes of 32, 64, 128, 256. What happens?

2. **Use a more complex dataset**: Replace digits with MNIST. Will deeper networks help more?

3. **Try different activation functions**: Replace ReLU with tanh or sigmoid. How does this affect optimal depth?

4. **Add regularization (L2)**: Use sklearn's `alpha` parameter to prevent overfitting. Does this change the optimal depth?

5. **Track convergence speed**: Why do some networks converge faster than others?